# ⚙️ Configuration & Export

How to configure parsing, mask proper nouns, and build multi-project datasets.

## 🎯 Goals of this notebook

1. **Understand RunConfig options** — learn every toggle the parser exposes
2. **Compare outputs** — see how dropping damaged signs or masking POS changes transliterations
3. **Build a multi-project dataset** — parse several corpora and combine into one DataFrame
4. **Export** — save combined data as JSONL and CSV for downstream use
5. **Quick analysis** — explore the combined dataset by project, provenance, and period

## Configuration Options

Before we start, if the default data paths aren't correct for your machine, you can re-configure them:
By default, `DATA_DIR` is set to `<repo_root>/enriched_data`. If you have it somewhere else (like `D:/thesis_data`):

In [1]:
import oracc_parser.settings as settings
from pathlib import Path

# settings.DATA_DIR = Path("D:/my_custom_oracc_data")

## 1. RunConfig options

| Option | Default | What it does |
|---|---|---|
| `limit` | `None` | Only parse first N texts (good for testing) |
| `max_break_fraction` | `1.0` | **Word-level**: fraction of broken signs tolerated before a word is replaced with `X` in transliteration / normalization / lemmatization |
| `drop_missing` | `False` | **Sign-level**: replace signs marked `[x]` (completely broken) with `X` in **Unicode cuneiform output only** |
| `drop_damaged` | `False` | **Sign-level**: replace signs marked `⸢x⸣` (partially damaged) with `X` in **Unicode cuneiform output only** |
| `mask_pos` | `[]` | Replace words of certain POS with the tag name |
| `languages` | `["Akkadian"]` | Which languages to process |
| `USE_CACHE` | `True` | Use cached results if available |

### ✅ Valid Config Values

When configuring a parser run with `RunConfig` (as seen in Notebook 01), you can filter by `languages` or `mask_pos`. The available values are exactly those listed in the reference tables below:

- **`languages`**: Any language listed in the `Languages` table (e.g., `Akkadian`, `Sumerian`, `Hittite`, `Ugaritic`, or simply `all` to include everything).
- **`mask_pos`**: Any POS tag from the `pos_tags` table below (e.g., `PN`, `DN`, `RN`, `N`, `V`).
- **`provenance` / `period`**: If filtering results programmatically by these fields in your analysis, use the `normalized_city` from the provenances table or the `period_name` from the periods table.

If you want to know what values are valid for `mask_pos`, `languages`, or `periods`, you can query the bundled reference data directly:

In [2]:
from oracc_parser.pipeline import reference_data

# See valid POS tags for masking
pos_df = reference_data.get_pos_tags()
print('Valid POS tags (first 15):')
print(pos_df['POS-tag'].head(15).tolist())
print()

# See valid languages
lang_df = reference_data.get_languages()
print('Valid Languages:')
print(lang_df['lang'].tolist())

Valid POS tags (first 15):
['MN', 'n', 'u', 'N', 'CN', 'AJ', 'PRP', 'V', 'IP', 'PN', 'MOD', 'RN', 'GN', 'REL', 'AV']

Valid Languages:
['sux', 'akk', 'akk-x-neoass', 'akk-x-stdbab', 'akk-x-neobab', 'akk-x-midass', 'akk-x-mbperi', 'qpc', 'akk-x-ltebab', 'akk-x-oldbab', 'akk-949', 'uga-040', 'sux-x-emesal', 'xur', 'peo', 'xur-946', 'hit', 'akk-x-midbab', 'arc', 'arc-949', 'elx', 'akk-x-stdbab-949', 'akk-x-earakk', 'akk-x-oldass', 'uga', 'xhu', 'hit-946', 'akk-936', 'xur-944', 'qca', 'qeb', 'akk-x-mbperi-949', 'akk-x-oldakk', 'qcu', 'qpe', 'grc', 'qur', 'akk-935', 'xhu-040', 'akk-x-neoass-949', 'xhu-946', 'sux-947', 'egy-020', 'sux-x-gloss', 'hlu', 'akk-x-neobab-949', 'sux-x-syll', 'hit-040', 'akk-x-oldbab-949']


> ⚠️ Important note: not all ORACC projects have POS-tags for every word in their corpus. When unknown, this field remains empty.

## 2. Word-level vs sign-level break filtering

`RunConfig` exposes **two independent levels** of break filtering that operate on different granularities and affect different outputs.

### Word-level — `max_break_fraction` (0.0 – 1.0)

- Each word has a `break_perc`: the **fraction of its signs** that are broken or missing (averaged across all signs in the word).

$\text{break\_perc} = \frac{n_{\text{broken}} + 0.5 \cdot n_{\text{damaged}}}{n_{\text{total}}}$

- Words whose `break_perc` **exceeds** `max_break_fraction` are replaced wholesale with `X`.
- Affects → **transliteration**, **normalization**, **lemmatization**.
- `1.0` (default) → keep all words regardless of damage.
- `0.0` → replace any word that has even one broken sign.

### Sign-level — `drop_missing` / `drop_damaged`

- Operates **sign-by-sign**, not word-by-word.
- `drop_missing=True` replaces signs marked `[x]` (completely lost) with `X`.
- `drop_damaged=True` replaces signs marked `⸢x⸣` (partially legible) with `X`.
- Affects → **Unicode cuneiform output only**.

> ⚠️ **The two levels are independent and produce results that may not align.**
> A word kept intact in the transliteration (because its *average* damage is below `max_break_fraction`) may still have individual signs stripped from the Unicode cuneiform output by `drop_missing` / `drop_damaged`, and vice-versa. Do not expect the text outputs and the Unicode cuneiform to be aligned token-for-token when using these parameters.

In [4]:
# max_break_fraction operates on the word level only
# It has no effect on Unicode cuneiform
from oracc_parser import parse_project_from_word_csvs, RunConfig, get_transliterations, get_unicode_texts
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR

PROJECT = "saao/saa01"
project_csv_dir = WORD_CSV_DIR / PROJECT.replace("/", "-")

# Load a 2-text sample
all_word_dfs = load_word_csvs_from_dir(project_csv_dir, project=PROJECT)
sample_dfs = dict(list(all_word_dfs.items())[:2])

# Strict word-level filtering: words with >30% broken signs → replaced with X
rec_strict = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(max_break_fraction=0.3))

# Liberal (default): keep all words regardless of damage
rec_liberal = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(max_break_fraction=1.0))

print("=== Transliteration with max_break_fraction=0.3 (strict) ===")
for _, r in get_transliterations(rec_strict).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Transliteration with max_break_fraction=1.0 (liberal, default) ===")
for _, r in get_transliterations(rec_liberal).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Unicode cuneiform is unaffected by max_break_fraction ===")
print("(drop_missing / drop_damaged control sign-level filtering here)")
for _, r in get_unicode_texts(rec_strict).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

12:59:18 | INFO    | oracc_parser | Loaded 264 word CSV(s) from G:\My Drive\GitHub\oracc-parser\enriched_data\oracc_csvs\saao-saa01
Processing saao/saa01: 100%|██████████| 2/2 [00:00<00:00,  4.43it/s]
12:59:18 | INFO    | oracc_parser | Processed 2 tablets for saao/saa01 from word CSVs
Processing saao/saa01: 100%|██████████| 2/2 [00:00<00:00,  6.15it/s]
12:59:19 | INFO    | oracc_parser | Processed 2 tablets for saao/saa01 from word CSVs


=== Transliteration with max_break_fraction=0.3 (strict) ===
  saao/saa01_P224395:
    a-na\t X EN-⸢ia⸣
ARAD-ka {1}10—⸢ha⸣-ti
X DI-mu X X EN⸣-ia
X {1}⸢a⸣-mi-ri
ina ŠA₃-bi X X {anše}a-na-qa-te
u₂-za-ta-⸢ki⸣ m
    tokens_without_broken: 46/131
  saao/saa01_P224403:
    a-bat LUGAL a-na\t X
07 me {tug₂m}ma-qar-rat ša\yd {še}IN.NU
07 me e-bi-is-su ša\yd X
ša\y 01-et e-bi-is-si
⸢ANŠE⸣.NITA₂
    tokens_without_broken: 29/34

=== Transliteration with max_break_fraction=1.0 (liberal, default) ===
  saao/saa01_P224395:
    a-na\t ⸢LUGAL⸣ EN-⸢ia⸣
ARAD-ka {1}10—⸢ha⸣-ti
⸢lu⸣ DI-mu ⸢a-na\t LUGAL EN⸣-ia
⸢DUMU⸣ {1}⸢a⸣-mi-ri
ina ŠA₃-bi 03 me⸣ {anše
    tokens_without_broken: 123/131
  saao/saa01_P224403:
    a-bat LUGAL a-na\t {lu₂v}⸢šak⸣-[ni]
07 me {tug₂m}ma-qar-rat ša\yd {še}IN.NU
07 me e-bi-is-su ša\yd {gi}⸢AMBAR-MEŠ⸣
ša\y 
    tokens_without_broken: 34/34

=== Unicode cuneiform is unaffected by max_break_fraction ===
(drop_missing / drop_damaged control sign-level filtering here)
  saao/saa01_P224

In [5]:
# drop_missing / drop_damaged operate sign-by-sign on Unicode cuneiform only
# They have no effect on transliteration, normalization, or lemmatization
rec_drop_none    = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(drop_missing=False, drop_damaged=False))
rec_drop_missing = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(drop_missing=True,  drop_damaged=False))
rec_drop_both    = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(drop_missing=True,  drop_damaged=True))

from oracc_parser import get_normalizations

for label, recs in [
    ("drop_missing=False, drop_damaged=False (default)", rec_drop_none),
    ("drop_missing=True",                               rec_drop_missing),
    ("drop_missing=True, drop_damaged=True",            rec_drop_both),
]:
    print(f"=== {label} ===")
    r = get_unicode_texts(recs).iloc[1]
    print(f"  Unicode:        {r['unicode'][:120]}")
    r2 = get_normalizations(recs).iloc[1]
    print(f"  Normalization:  {r2['normalization'][:120]}  (unchanged)")
    print()


Processing saao/saa01: 100%|██████████| 2/2 [00:00<00:00,  5.45it/s]
12:59:24 | INFO    | oracc_parser | Processed 2 tablets for saao/saa01 from word CSVs
Processing saao/saa01: 100%|██████████| 2/2 [00:00<00:00,  5.49it/s]
12:59:24 | INFO    | oracc_parser | Processed 2 tablets for saao/saa01 from word CSVs
Processing saao/saa01: 100%|██████████| 2/2 [00:00<00:00,  6.18it/s]
12:59:24 | INFO    | oracc_parser | Processed 2 tablets for saao/saa01 from word CSVs


=== drop_missing=False, drop_damaged=False (default) ===
  Unicode:        𒀀𒁁 𒈗 𒀀𒈾 𒊕𒉌
𒐌 𒈨 𒌆𒈠𒃼𒋥 𒊭 𒊺𒅔𒉡
𒐌 𒈨 𒂊𒁉𒄑𒋢 𒊭 𒄀𒆹𒎌
𒊭 𒁹𒀉 𒂊𒁉𒄑𒋛
𒀲𒀴 𒆷 𒂊𒈬𒋡𒋙𒌋𒉌
𒆷 𒄿𒆳𒋫𒄷𒌋𒉌
𒌓 𒁹𒄰 𒊭 𒌗𒃶
𒀸 𒌷𒂦𒎙𒁺
𒇻 𒄥𒁍
𒁹𒂗 𒌓𒈬 𒂊𒋼𒋾𒅅
𒋫𒈬𒀜
  Normalization:  abat šarri ana šakni
UNK mē maqarrāt ša tibni
UNK mē ebissu ša appārī
ša issēt ebissi
imāru lā emūqašūni
lā imattahūni
ū  (unchanged)

=== drop_missing=True ===
  Unicode:        𒀀𒁁 𒈗 𒀀𒈾 𒊕X
𒐌 𒈨 𒌆𒈠𒃼𒋥 𒊭 𒊺𒅔𒉡
𒐌 𒈨 𒂊𒁉𒄑𒋢 𒊭 𒄀𒆹𒎌
𒊭 𒁹𒀉 𒂊𒁉𒄑𒋛
𒀲𒀴 𒆷 𒂊𒈬𒋡𒋙𒌋𒉌
𒆷 𒄿𒆳𒋫𒄷𒌋𒉌
𒌓 𒁹𒄰 𒊭 𒌗𒃶
𒀸 𒌷𒂦𒎙𒁺
𒇻 𒄥𒁍
𒁹𒂗 𒌓𒈬 XX𒋾𒅅
𒋫𒈬𒀜
  Normalization:  abat šarri ana šakni
UNK mē maqarrāt ša tibni
UNK mē ebissu ša appārī
ša issēt ebissi
imāru lā emūqašūni
lā imattahūni
ū  (unchanged)

=== drop_missing=True, drop_damaged=True ===
  Unicode:        𒀀𒁁 𒈗 𒀀𒈾 XX
𒐌 𒈨 𒌆𒈠𒃼𒋥 𒊭 𒊺𒅔𒉡
𒐌 𒈨 𒂊𒁉𒄑𒋢 𒊭 𒄀XX
𒊭 𒁹𒀉 𒂊𒁉𒄑𒋛
X𒀴 𒆷 𒂊𒈬𒋡𒋙𒌋X
𒆷 𒄿𒆳𒋫𒄷𒌋𒉌
𒌓 𒁹X 𒊭 𒌗𒃶
𒀸 XX𒎙𒁺
𒇻 XX
𒁹𒂗 𒌓X XX𒋾X
𒋫XX
  Normalization:  abat šarri ana šakni
UNK mē maqarrāt ša tibni
UNK mē ebissu ša appārī
ša issēt ebissi
imāru lā emūqašūni
lā imattahūni
ū  (unchanged)



## 3. Build a multi-project dataset

In [6]:
# Parse a few projects and combine
import pandas as pd
from oracc_parser import parse_project_from_word_csvs, RunConfig, get_full_flat_table, get_full_flat_table, get_metadata_table
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR, OUTPUT_DIR

PROJECTS = ["saao/saa01", "saao/saa05"]  # change these to what you want
config = RunConfig(max_break_fraction=.5, drop_missing=True)

all_dfs = []
for project in PROJECTS:
    print(f"Parsing {project}...")
    try:
        project_csv_dir = WORD_CSV_DIR / project.replace("/", "-")
        word_dfs = load_word_csvs_from_dir(project_csv_dir, project=project)
        # Limit to first 3 texts for demo; remove slice for full parse
        sample_dfs = dict(list(word_dfs.items())[:3])
        records = parse_project_from_word_csvs(project, sample_dfs, config=config)
        df = get_full_flat_table(records)
        all_dfs.append(df)
        print(f"  -> {len(records)} tablets")
    except Exception as e:
        print(f"  -> Error: {e}")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"Combined: {len(combined)} tablets from {len(PROJECTS)} projects")
display(combined)

Parsing saao/saa01...


12:59:35 | INFO    | oracc_parser | Loaded 264 word CSV(s) from G:\My Drive\GitHub\oracc-parser\enriched_data\oracc_csvs\saao-saa01
Processing saao/saa01: 100%|██████████| 3/3 [00:00<00:00,  5.16it/s]
12:59:36 | INFO    | oracc_parser | Processed 3 tablets for saao/saa01 from word CSVs


  -> 3 tablets
Parsing saao/saa05...


12:59:38 | INFO    | oracc_parser | Loaded 298 word CSV(s) from G:\My Drive\GitHub\oracc-parser\enriched_data\oracc_csvs\saao-saa05
Processing saao/saa05: 100%|██████████| 3/3 [00:00<00:00,  5.61it/s]
12:59:39 | INFO    | oracc_parser | Processed 3 tablets for saao/saa05 from word CSVs


  -> 3 tablets
Combined: 6 tablets from 2 projects


,id,project,text_id,genre,archive,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,saao/saa01_P224395,saao/saa01,P224395,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,Neo-Assyrian,-721,-705,a-na\t ⸢LUGAL⸣ EN-⸢ia⸣\nARAD-ka {1}10—⸢ha⸣-ti\...,ana šarri bēlīya\nurdaka Adda-hati\nlū šulmu a...,ana šarru bēlu\nardu Adda-hati\nlū šulmu ana š...,𒀀𒈾 𒈗 𒂗𒅀\n𒀴𒅗 𒁹𒌋𒄩𒋾\n𒇻 𒁲𒈬 𒀀𒈾 𒈗 𒂗𒅀\n𒌉 𒁹𒀀𒈪𒊑\n𒀸 𒊮𒁉 𒐈...,"To the king, my lord: Your servant Adda-hati. ...",131,75
1,saao/saa01_P224403,saao/saa01,P224403,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,Neo-Assyrian,-717,-706,a-bat LUGAL a-na\t {lu₂v}⸢šak⸣-[ni]\n07 me {tu...,abat šarri ana šakni\nUNK mē maqarrāt ša tibni...,awātu šarru ana šaknu\nUNK meʾatu maqarratu ša...,𒀀𒁁 𒈗 𒀀𒈾 𒊕X\n𒐌 𒈨 𒌆𒈠𒃼𒋥 𒊭 𒊺𒅔𒉡\n𒐌 𒈨 𒂊𒁉𒄑𒋢 𒊭 𒄀𒆹𒎌\n𒊭...,The king's word to the go[vernor] (of Calah):\...,34,33
2,saao/saa01_P224417,saao/saa01,P224417,Administrative Letter,Uncertain (North-West Palace Room),Nimrud,Neo-Assyrian,-721,-705,a-na\t LUGAL EN-ia\nARAD\d-ka {1}10—ha-ti\t\n⸢...,ana šarri bēlīya\nurdaka Adda-hati\nlū šulmu a...,ana šarru bēlu\nardu Adda-hati\nlū šulmu ana š...,𒀀𒈾 𒈗 𒂗𒅀\n𒀴𒅗 𒁹𒌋𒄩𒋾\n𒇻 𒁲𒈬 𒀀𒈾 𒈗 𒂗𒅀\n𒆬𒌓 𒊭 𒃻𒉡𒈨𒋼 𒊭 ...,"To the king, my lord: Your servant Adda-hati. ...",193,140
3,saao/saa05_P224392,saao/saa05,P224392,Administrative Letter,Uncertain,Nimrud,Neo-Assyrian,-721,-705,a-na LUGAL be-li₂-ia\nARAD-ka {1}mah-de-e\nlu-...,ana šarri bēlīya\nurdaka Mahde\nlū šulmu ana š...,ana šarru bēlu\nardu Mahde\nlū šulmu ana šarru...,𒀀𒈾 𒈗 𒁁𒉌𒅀\n𒀴𒅗 𒁹𒈤𒁲𒂊\n𒇻𒌋 𒁲𒈬 𒀀𒈾 𒎙 𒂗𒅀\n𒆗𒇷𒌑 𒊭 𒉌𒁕𒉡𒉌\n...,"To the king, my lord: your servant Mahdê. Good...",77,74
4,saao/saa05_P224439,saao/saa05,P224439,Administrative Letter,Uncertain,Nimrud,Neo-Assyrian,-721,-705,X ⸢LUGAL⸣ EN-ia\nX {1}{d}IM—KI-ia\nX ⸢DI⸣-mu a...,X šarri bēlīya\nX Adad-isseʾa\nX šulmu ana šar...,X šarru bēlu\nX Adad-isseʾa\nX šulmu ana šarru...,XX 𒈗 𒂗𒅀\nXX 𒁹𒀭𒅎𒆠𒅀\nX 𒁲𒈬 𒀀𒈾 𒈗 𒂗𒅀\n𒊭 𒈗 𒂗 𒉈𒂊𒈬 𒅖𒆪𒈾...,"[To the ki]ng, my lord: [your servant] Adad-is...",131,126
5,saao/saa05_P224586,saao/saa05,P224586,Administrative Letter,Uncertain,Nimrud,Neo-Assyrian,-721,-705,X X X\nX X X X X X\n⸢lu⸣ X X X X\nšul-mu a-⸢na...,X X X\nX X X X X X\nlū X X X X\nšulmu ana X X ...,X X X\nX X X X X X\nlū X X X X\nšulmu ana X X ...,XX X XXX\nXX XX X X X X\n𒇻 XX XX X XX\n𒂄𒈬 𒀀𒈾 X...,"[To the king, my lord: your servant NN]. Go[od...",122,40


In [7]:
# Export the combined dataset
out = OUTPUT_DIR
out.mkdir(parents=True, exist_ok=True)

# change the file names here, if you want
path_jsonl = out / "combined_dataset.jsonl"
path_csv = out / "combined_dataset.csv"

combined.to_json(path_jsonl, orient="records", lines=True, force_ascii=False)
combined.to_csv(path_csv, index=False)

print(f"Saved to:")
print(f"  {path_jsonl}  ({path_jsonl.stat().st_size/1024:.1f} KB)")
print(f"  {path_csv}  ({path_csv.stat().st_size/1024:.1f} KB)")
print(f"\n📁 All output files in {out}:")
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Saved to:
  G:\My Drive\GitHub\oracc-parser\enriched_data\output\combined_dataset.jsonl  (25.7 KB)
  G:\My Drive\GitHub\oracc-parser\enriched_data\output\combined_dataset.csv  (23.9 KB)

📁 All output files in G:\My Drive\GitHub\oracc-parser\enriched_data\output:
  combined_dataset.csv                      23.9 KB
  combined_dataset.jsonl                    25.7 KB
  etcsri.csv                                3.2 KB
  etcsri.jsonl                              4.3 KB
  saao_saa01.csv                            26.8 KB
  saao_saa01.jsonl                          28.5 KB


## 5. Quick analysis of the data

In [8]:
print("Texts by project:")
print(combined["project"].value_counts().to_string())

print("\nTexts by provenance:")
print(combined["provenance"].value_counts().to_string())

print("\nTexts by period:")
print(combined["period"].value_counts().to_string())

print(f"\nAvg tokens per text: {combined['total_tokens'].mean():.0f}")
print(f"Total tokens: {combined['total_tokens'].sum()}")

Texts by project:
project
saao/saa01    3
saao/saa05    3

Texts by provenance:
provenance
Nimrud    6

Texts by period:
period
Neo-Assyrian    6

Avg tokens per text: 115
Total tokens: 688


## What's next?

- **Quickstart:** See `01_quickstart.ipynb` to quickly download data, parse a project from CSVs, and export results
- **Explore what's in the dataset:** See notebook `02_reference_data.ipynb` for a full overview of collected projects, catalogues, provenance, periods, and other reference data.
- **understand the process:** see notebook `04_oracc_json_processing.ipynb` for in-depth explanations on how the package processes the original oracc files and metadata.